## 将dicom序列转换成mhd进行存储
1. 生成混合电压图像的mhd
2. 生成剪影图像的mhd

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from collections import  Counter
import SimpleITK as sitk
import time
import nibabel as nib

In [2]:
series_uids = []
series_list_file = '/data/zhangwd/data/examples/brain/bone_removed/removed_dicom.txt'
with open(series_list_file) as f:
    for line in f.readlines():
        line = line.strip()
        if line is None or len(line) == 0:
            continue
        series_uids.append(line)
series_uid = series_uids[0]
print(series_uid)

/data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016072900171834200000026/1.3.12.2.1107.5.99.2.9594.30000016072913081812500000799


In [4]:
outdir = '/data/zhangwd/data/examples/brain/mhd/bone_removed'
os.makedirs(outdir,exist_ok=True)
for series_uid in series_uids:
    print('====> begin process {}'.format(series_uid))
    reader = sitk.ImageSeriesReader()
    filenamesDicom = reader.GetGDCMSeriesFileNames(series_uid)
    reader.SetFileNames(filenamesDicom)
    dicom_imgs = reader.Execute()
    basename = os.path.basename(series_uid)
    sitk.WriteImage(dicom_imgs, os.path.join(outdir, '{}.mhd'.format(basename)))
    print('====> end process {}\t'.format(series_uid))

====> begin process /data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016072900171834200000026/1.3.12.2.1107.5.99.2.9594.30000016072913081812500000799
====> end process /data/zhangwd/data/examples/brain/bystudy/1.3.12.2.1107.5.1.4.60320.30000016072900171834200000026/1.3.12.2.1107.5.99.2.9594.30000016072913081812500000799	
====> begin process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240
====> end process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20170320003131115/1.3.12.2.1107.5.99.2.9594.30000017032111384964000000240	
====> begin process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20181017005827621/1.3.12.2.1107.5.99.2.9594.30000018101817420531200000128
====> end process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20181017005827621/1.3.12.2.1107.5.99.2.9594.30000018101817420531200000128	
====> begin process /da

====> end process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20190514023531071/1.3.12.2.1107.5.99.2.9594.30000019051009454809300013399	
====> begin process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20160415001724515/1.3.12.2.1107.5.99.2.9594.30000016041520025798400003100
====> end process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20160415001724515/1.3.12.2.1107.5.99.2.9594.30000016041520025798400003100	
====> begin process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20160302001630238/1.3.12.2.1107.5.99.2.9594.30000016030213261042100008237
====> end process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.20160302001630238/1.3.12.2.1107.5.99.2.9594.30000016030213261042100008237	
====> begin process /data/zhangwd/data/examples/brain/bystudy/1.2.392.200046.100.14.2016072900451278/1.3.12.2.1107.5.99.2.9594.30000016072913081812500001978
====> end process /data/zhangwd/data/examples/brain/byst

In [10]:
from glob import glob
mhd_list = glob(os.path.join(outdir, '*.mhd'))
mhd_list = [os.path.basename(i) for i in mhd_list]
indir = '/home/zhangwd/mhd/bone_removed'
seg_out_dir = '/home/zhangwd/mhd/out/'
cmd_list = []
for mhd_file in mhd_list:
    cmd = './tubeSegmentation {} --parameters Neuro-Vessels-USA --storage-dir {} --storage-name {}|tee {}'.format(
        os.path.join(indir, mhd_file), seg_out_dir, mhd_file.replace('.mhd', ''), os.path.join(seg_out_dir, '{}.log'.format(mhd_file)))
    cmd_list.append(cmd)
with open('/data/zhangwd/data/examples/brain/mhd/neuro_vascular_seg_task.sh', 'w') as f:
    f.write('\n'.join(cmd_list))